In [ ]:
import numpy as np
import librosa
import joblib
import tensorflow as tf

# ==============================
# CONFIG
# ==============================

SR = 16000
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512
IMG_WIDTH = 128

audio_path = "/content/drive/MyDrive/mimii_fan/6_dB_fan/fan/id_02/abnormal/00000000.wav"


# ==============================
# LOAD FAN ID CLASSIFIER
# ==============================

id_classifier = joblib.load("fan_id_classifier.pkl")


# ==============================
# FEATURE EXTRACTION
# ==============================

def extract_features(y, sr):

    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    energy = np.sum(y**2)

    fft = np.fft.rfft(y)
    mag = np.abs(fft)
    freqs = np.fft.rfftfreq(len(y), 1/sr)

    mask = (freqs >= 1000) & (freqs <= 3000)

    freqs = freqs[mask]
    mag = mag[mask]

    dominant = freqs[np.argmax(mag)]

    return centroid, bandwidth, energy, dominant


# ==============================
# MEL SPECTROGRAM
# ==============================

def extract_mel(y):

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    if mel_db.shape[1] < IMG_WIDTH:
        pad = IMG_WIDTH - mel_db.shape[1]
        mel_db = np.pad(mel_db, ((0,0),(0,pad)), mode='constant')
    else:
        mel_db = mel_db[:, :IMG_WIDTH]

    return mel_db


# ==============================
# TFLITE FUNCTIONS
# ==============================

def load_tflite_model(path):
    interpreter = tf.lite.Interpreter(model_path=path)
    interpreter.allocate_tensors()
    return interpreter


def predict_tflite(interpreter, input_data):

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data.astype(np.float32))
    interpreter.invoke()

    return interpreter.get_tensor(output_details[0]['index'])


# ==============================
# LOAD AUDIO
# ==============================

y, sr = librosa.load(audio_path, sr=SR)

centroid, bandwidth, energy, dom = extract_features(y, sr)

# IMPORTANT: order must match training
id_features = np.array([[centroid, bandwidth, energy, dom]])

predicted_id = id_classifier.predict(id_features)[0]

print("\nDetected Fan ID:", predicted_id)


# ==============================
# LOAD MODELS
# ==============================

# TFLite CAE
interpreter = load_tflite_model(f"model_{predicted_id}.tflite")

# Fault classifier
fault_model = joblib.load(f"{predicted_id}_fault_model.pkl")
threshold = np.load(f"threshold_{predicted_id}.npy").item()

# Normalization
train_min = np.load(f"train_min_{predicted_id}.npy").item()
train_max = np.load(f"train_max_{predicted_id}.npy").item()


# ==============================
# AUTOENCODER INPUT
# ==============================

mel = extract_mel(y)

mel = (mel - train_min) / (train_max - train_min + 1e-8)

mel = mel[..., np.newaxis]
mel = np.expand_dims(mel, axis=0)


# ==============================
# RECONSTRUCTION (TFLITE)
# ==============================

recon = predict_tflite(interpreter, mel)

reconstruction_error = np.mean(np.abs(mel - recon))

print("Reconstruction Error:", reconstruction_error)


# ==============================
# FAULT CLASSIFIER
# ==============================

fault_features = np.array([[
    reconstruction_error,
    centroid,
    bandwidth,
    dom
]])

fault_prob = fault_model.predict_proba(fault_features)[0][1]

#print("Fault Probability:", fault_prob)


# ==============================
# FINAL RESULT
# ==============================

if fault_prob > threshold:
    print(f"\n⚠️ RESULT: FAN {predicted_id} IS ABNORMAL")
else:
    print(f"\n✅ RESULT: FAN {predicted_id} IS NORMAL")


Detected Fan ID: id_02
Reconstruction Error: 0.044397704

⚠️ RESULT: FAN id_02 IS ABNORMAL


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
